# File 05 — ML Analytics + Validation
**EHSL-ML+ | Disertasi Nanan Soekarna**

File ini menjalankan 3 tugas machine learning inti (Layer 3 EHSL-ML+), sesuai proposal §ML Analytics:

1. **Clustering** — kelompokkan responden dgn pola persepsi mirip (K-Means + Hierarchical)
2. **Classification** — 3 himpunan fitur yang dibedakan secara tegas (lihat di bawah)
3. **Anomaly Detection** — deteksi pola skor tidak wajar (Isolation Forest + LOF + baseline statistik)

Ditambah **validasi** (repeated stratified CV, confusion matrix, perbandingan model) dan implementasi **custom objective function** `L_total`.

## Tiga isu ML penting di file ini

**Isu 1 — Data Leakage bersifat STRUKTURAL, bukan sekadar beberapa kolom.** Target (kategori NREI) dihitung dari skor VfV/FCNC/ID/GR. Masalahnya, hampir semua fitur di `feature_matrix` adalah **komponen** dari skor-skor itu: kolom `ECV_{aktor}` adalah skor per-aktor mentah, dan skor keseluruhan hanyalah rata-ratanya. Jadi membuang kolom `MAS/ECI/NREI` saja **tidak menutup kebocoran** — skor asli masih bisa dihitung balik dengan merata-ratakan kolom ECV, atau secara aljabar dari fitur `IF_*` (perkalian skor). Sudah diverifikasi: korelasi rekonstruksi = 1.000000, max error = 0.0.

Konsekuensinya, satu himpunan fitur "aman" saja tidak cukup. Di bawah dipakai **3 himpunan yang diberi label jujur**:

| Himpunan | Isi | Boleh disebut |
|---|---|---|
| **SET A** | semua fitur, termasuk skor agregat | demonstrasi kebocoran telanjang |
| **SET B** | skor agregat & `IF_*` dibuang, komponen granular tetap | **pemeriksaan konsistensi internal** — BUKAN prediksi |
| **SET C** | hanya demografi/legal, nol komponen target | **prediksi** (satu-satunya yang sah) |

Kenapa SET B tetap berguna meski bocor? Karena pertanyaannya berbeda: bukan "bisakah kita memprediksi NREI" (tidak bisa, itu sirkular), tapi "seberapa koheren kategori NREI dengan pola jawaban granularnya, dan komponen granular mana yang paling menentukan" — itu pertanyaan yang sah dan berguna untuk SHAP di File 06. Yang penting: jangan dilaporkan sebagai kemampuan prediktif.

**Isu 2 — Imputasi tidak boleh mengintip data test.** Versi sebelumnya menjalankan `X.fillna(X.mean())` pada seluruh data sebelum split, sehingga rata-rata pengisi ikut dihitung dari data test. Sekarang imputasi dipasang di dalam `Pipeline` sklearn supaya di-fit hanya pada fold training.

**Isu 3 — Custom Objective `L_total = L_classification + λ₁·L_fairness + λ₂·L_ethical`.** Dua kendala teknis: (a) Random Forest tidak bisa dikasih loss kustom (pakai Gini, bukan gradient descent); (b) `L_ethical` sebagaimana ditulis cuma bergantung pada nilai fitur, bukan prediksi model — jadi tidak bisa jadi gradien pelatihan. Solusi jujur: kita hitung `L_fairness`/`L_ethical` sebagai **metrik evaluasi + sample-weighting**, bukan pura-pura sebagai custom gradient. Semua komponen dihitung dari **satu model yang sama**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, RepeatedStratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, silhouette_score, adjusted_rand_score, log_loss)
from scipy import stats
import xgboost as xgb

np.random.seed(42)

fm = pd.read_csv("feature_matrix.csv")
idx = pd.read_csv("data_index_respondent.csv")
target = idx[["respondent_id","NREI_overall_cat3"]].rename(columns={"NREI_overall_cat3":"NREI_cat"})
df = fm.merge(target, on="respondent_id", how="inner").dropna(subset=["NREI_cat"]).reset_index(drop=True)

actor_list = ["Investigator","Prosecutor","Judge","Advocate","CorrectionalOfficer"]

# Flag straight-lining dari File 00 akhirnya DIPAKAI (sebelumnya dibuat lalu
# diabaikan). Dipakai untuk sensitivity check di bawah, bukan untuk membuang
# data secara sepihak.
try:
    _ia = pd.read_csv("data_index_actor.csv")
    _sl = _ia.groupby("respondent_id")["flag_straightline"].any().rename("flag_straightline")
    df = df.merge(_sl, on="respondent_id", how="left")
    df["flag_straightline"] = df["flag_straightline"].fillna(False).astype(bool)
    print(f"Responden dgn >=1 baris straight-lining: {int(df['flag_straightline'].sum())}")
except Exception as e:
    df["flag_straightline"] = False
    print(f"(flag straight-lining tidak tersedia: {e})")

print(f"N responden: {len(df)}")
print(f"Distribusi target NREI_cat: {dict(df['NREI_cat'].value_counts())}")

Responden dgn >=1 baris straight-lining: 11
N responden: 199
Distribusi target NREI_cat: {'Moderate': np.int64(108), 'Low': np.int64(50), 'High': np.int64(41)}


## 1. Clustering (Unsupervised)

Clustering itu **unsupervised** — tidak ada "kunci jawaban" (target), jadi tidak ada risiko leakage. Tujuannya: menemukan apakah responden secara alami membentuk kelompok-kelompok pola persepsi (misal "kelompok yang menganggap governance tinggi di semua aktor" vs "kelompok yang rendah di tahap penyidikan saja").

Jumlah cluster optimal dicari lewat **silhouette score** — ukuran seberapa "rapi" pemisahan cluster (makin dekat 1, makin rapi).

In [2]:
cluster_features = ["VfV_overall","FCNC_overall","ID_overall","MAS_overall","ECI_overall","NREI_overall"]
Xc = df[cluster_features].fillna(df[cluster_features].mean())
Xc_scaled = StandardScaler().fit_transform(Xc)

sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xc_scaled)
    sil_scores[k] = silhouette_score(Xc_scaled, labels)
    print(f"k={k}: silhouette={sil_scores[k]:.3f}, WCSS={km.inertia_:.1f}")

best_k = max(sil_scores, key=sil_scores.get)
print(f"-> k optimal: {best_k}")

k=2: silhouette=0.506, WCSS=460.7


k=3: silhouette=0.390, WCSS=302.2
k=4: silhouette=0.405, WCSS=201.8
k=5: silhouette=0.356, WCSS=165.1
k=6: silhouette=0.331, WCSS=141.4
-> k optimal: 2


In [3]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["cluster_kmeans"] = km.fit_predict(Xc_scaled)
hier = AgglomerativeClustering(n_clusters=best_k)
df["cluster_hierarchical"] = hier.fit_predict(Xc_scaled)

ari = adjusted_rand_score(df["cluster_kmeans"], df["cluster_hierarchical"])
print(f"Kesepakatan K-Means vs Hierarchical (Adjusted Rand Index): {ari:.3f}")
print("(1.0 = identik, 0 = acak. Makin tinggi makin yakin cluster-nya nyata, bukan artefak 1 algoritma)")

# Uji kestabilan eksplisit: ARI rendah = dua algoritma tidak sepakat = cluster
# TIDAK boleh dilaporkan sebagai tipologi responden.
AMBANG_ARI = 0.50
cluster_stabil = ari >= AMBANG_ARI
if cluster_stabil:
    print(f"-> ARI >= {AMBANG_ARI}: cluster dinilai STABIL, layak diinterpretasi.")
else:
    print(f"-> ARI < {AMBANG_ARI}: kedua algoritma TIDAK sepakat soal keanggotaan cluster.")
    print("   Cluster ini dinilai TIDAK STABIL. Profil di bawah tetap dicetak sebagai")
    print("   deskripsi, TAPI tidak boleh dilaporkan sebagai tipologi/temuan di Bab IV.")

profile = df.groupby("cluster_kmeans")[cluster_features].mean().round(1)
profile["n"] = df.groupby("cluster_kmeans").size()
print("\nProfil tiap cluster (rata-rata skor):")
print(profile.to_string())

Kesepakatan K-Means vs Hierarchical (Adjusted Rand Index): 0.184
(1.0 = identik, 0 = acak. Makin tinggi makin yakin cluster-nya nyata, bukan artefak 1 algoritma)
-> ARI < 0.5: kedua algoritma TIDAK sepakat soal keanggotaan cluster.
   Cluster ini dinilai TIDAK STABIL. Profil di bawah tetap dicetak sebagai
   deskripsi, TAPI tidak boleh dilaporkan sebagai tipologi/temuan di Bab IV.

Profil tiap cluster (rata-rata skor):
                VfV_overall  FCNC_overall  ID_overall  MAS_overall  ECI_overall  NREI_overall    n
cluster_kmeans                                                                                    
0                      64.3          64.8        65.1         64.5         64.9          62.1   83
1                      37.8          38.2        38.4         38.0         38.3          40.2  116


**Cara menafsirkan profil cluster:** lihat pola rata-rata skornya. Cluster dgn semua skor tinggi = "persepsi governance baik menyeluruh". Cluster dgn FCNC rendah tapi VfV tinggi = "merasa diperlakukan manusiawi tapi curiga prosedurnya tidak jujur". Ingat pesan proposal: cluster ini adalah **pola persepsi**, BUKAN bukti pelanggaran oleh aktor/institusi manapun.

## 2. Classification — Tiga Himpunan Fitur dengan Label Jujur

Sesuai pembahasan Isu 1 di atas, kita definisikan 3 himpunan dan **audit kebocoran masing-masing secara langsung** (bukan diasumsikan):

- **SET A** — semua fitur, termasuk `MAS/ECI/NREI`. Bocor telanjang.
- **SET B** — skor agregat dan `IF_*` dibuang. Tetap bocor secara struktural (komponen granular = bahan target), jadi dilabeli **konsistensi internal**, bukan prediksi.
- **SET C** — hanya demografi/legal (usia, gender, pendidikan, jenis kasus, lama vonis, masa tahanan, lapas, wilayah). Nol komponen target → satu-satunya yang sah disebut **prediksi**.

Audit kebocoran di bawah mencoba menghitung balik skor konstruk dari fitur yang tersedia. Kalau korelasinya ~1.0, berarti kunci jawaban ikut bocor.

**Soal metrik:** angka headline sekarang dari **repeated stratified CV** (5 lipatan × 10 ulangan = 50 evaluasi) beserta interval, bukan satu kali split. Alasannya: test set dari split tunggal hanya 40 orang, sehingga selisih antar model setara ~1 sampel dan tidak bisa dijadikan dasar klaim. Sebagai patokan, **baseline kelas mayoritas** juga dihitung — model yang tidak mengalahkan baseline berarti tidak menemukan apa pun.

In [4]:
label_map = {"Low":0, "Moderate":1, "High":2}
y = df["NREI_cat"].map(label_map).values

exclude_nonfeature = ["respondent_id","NREI_cat","cluster_kmeans","cluster_hierarchical",
                      "flag_straightline"]
all_feature_cols = [c for c in df.columns if c not in exclude_nonfeature]

# ---------------------------------------------------------- definisi 3 himpunan
# (1) Skor agregat: langsung menyusun target.
agg_cols = ["VfV_overall","FCNC_overall","ID_overall","GR_overall",
            "MAS_overall","ECI_overall","NREI_overall",
            "NREI_Investigator","NREI_Prosecutor","NREI_Judge",
            "NREI_Advocate","NREI_CorrectionalOfficer"]
# (2) IF_* = fungsi dari skor agregat, jadi skor asli bisa dihitung balik:
#       VfV = 100*sqrt(IF_VfV_FCNC * IF_VfV_ID / IF_FCNC_ID)
#     IF_ID_Stage = ID_overall/100 * n_aktor/5 -> versi terskala dari ID_overall,
#     dan ia sempat mendominasi SHAP sampai menutupi fitur granular.
#     Penggantinya yang bersih: n_actors_encountered (dipertahankan di File 04).
if_cols = ["IF_VfV_FCNC","IF_FCNC_ID","IF_VfV_ID","IF_ID_Stage"]
# (3) Kolom pivot VfV_{aktor} TERBUKTI byte-identik dgn ECV_{aktor}_VfV.
#     Fitur kembar tidak menambah informasi, tapi memecah/mengencerkan nilai
#     SHAP sehingga peringkat fitur jadi sulit dibaca. Satu salinan dibuang.
dup_cols = [f"VfV_{a}" for a in actor_list]

set_a_cols = all_feature_cols
set_b_cols = [c for c in all_feature_cols if c not in agg_cols + if_cols + dup_cols]
set_c_cols = ["age","gender_encoded","education_encoded","case_type_encoded",
              "sentence_length_months","time_served_months",
              "institution_encoded","region_encoded"]

def ambil_X(cols):
    return df[[c for c in cols if c in df.columns]].select_dtypes(include=[np.number])

X_a, X_b, X_c = ambil_X(set_a_cols), ambil_X(set_b_cols), ambil_X(set_c_cols)
print(f"SET A (bocor telanjang)      : {X_a.shape[1]} fitur")
print(f"SET B (konsistensi internal) : {X_b.shape[1]} fitur"
      f"  [{len(dup_cols)} kolom kembar & {len(if_cols)} kolom IF_* dibuang]")
print(f"SET C (prediksi bersih)      : {X_c.shape[1]} fitur")
print(f"n_actors_encountered ikut di SET B: {'n_actors_encountered' in X_b.columns}")
print(f"\nRasio fitur:responden SET B = {X_b.shape[1]}:{len(df)} "
      f"= 1:{len(df)/max(X_b.shape[1],1):.1f}  (makin kecil makin rawan overfitting)")

# ------------------------------------------------- audit kebocoran tiap himpunan
def audit_kebocoran(nama, X):
    """Coba hitung balik skor konstruk dari fitur yg diberikan ke model.
    Korelasi mendekati 1.0 = kunci jawaban bocor."""
    jalur = []
    if all(c in X.columns for c in ["IF_VfV_FCNC","IF_FCNC_ID","IF_VfV_ID"]):
        a, b, c = X["IF_VfV_FCNC"], X["IF_FCNC_ID"], X["IF_VfV_ID"]
        with np.errstate(divide="ignore", invalid="ignore"):
            jalur.append(("aljabar IF_*", np.sqrt(a*c/b)*100, "VfV"))
    for con in ["VfV","FCNC","ID"]:
        ecv = [f"ECV_{a}_{con}" for a in actor_list if f"ECV_{a}_{con}" in X.columns]
        if ecv:
            jalur.append((f"rata2 ECV_{{aktor}}_{con}", X[ecv].mean(axis=1), con))
    for con in ["VfV","FCNC","ID"]:
        if f"{con}_overall" in X.columns:
            jalur.append((f"{con}_overall langsung", X[f"{con}_overall"], con))
    if not jalur:
        print(f"  {nama:28s}: tidak ada jalur rekonstruksi -> BERSIH")
        return 0.0
    tertinggi = 0.0
    for label, rec, con in jalur:
        r = pd.Series(np.asarray(rec), index=X.index).corr(df[f"{con}_overall"])
        tertinggi = max(tertinggi, abs(r))
        tanda = "  <-- BOCOR" if abs(r) >= 0.90 else ""
        print(f"  {nama:28s}: {label:24s} korelasi={r:.6f}{tanda}")
    return tertinggi

print("\n=== AUDIT KEBOCORAN ===")
r_a = audit_kebocoran("SET A", X_a)
r_b = audit_kebocoran("SET B", X_b)
r_c = audit_kebocoran("SET C", X_c)
print(f"\nKorelasi rekonstruksi tertinggi -> SET A={r_a:.4f}  SET B={r_b:.4f}  SET C={r_c:.4f}")
assert r_c < 0.90, "SET C seharusnya bersih dari komponen target!"
assert not any(c in X_b.columns for c in if_cols), "IF_* seharusnya sudah keluar dari SET B!"
assert not any(c in X_b.columns for c in dup_cols), "kolom kembar seharusnya sudah keluar!"
print("OK: assert lolos -> SET C bersih, SET B bebas fitur IF_* dan kolom kembar.")

# --------------------------------- evaluasi: imputasi DI DALAM Pipeline + repeated CV
def buat_pipe(model, scale=False, k_select=None):
    """Imputasi (dan scaling / seleksi fitur) di dalam Pipeline -> di-fit HANYA
    pada fold training, tidak pernah mengintip data test."""
    steps = [("imputer", SimpleImputer(strategy="mean"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    if k_select:
        steps.append(("select", SelectKBest(f_classif, k=k_select)))
    steps.append(("model", model))
    return Pipeline(steps)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

def evaluasi(nama, X, k_select=None, y_vec=None):
    yy = y if y_vec is None else y_vec
    pipe = buat_pipe(RandomForestClassifier(n_estimators=200, random_state=42,
                                            class_weight="balanced"), k_select=k_select)
    s = cross_val_score(pipe, X, yy, cv=cv, scoring="accuracy", n_jobs=-1)
    print(f"{nama:34s} akurasi = {s.mean():.3f} +/- {s.std():.3f}   "
          f"[p2.5-p97.5: {np.percentile(s,2.5):.3f}-{np.percentile(s,97.5):.3f}]")
    return s

print("\n=== AKURASI (repeated stratified CV, 5 lipatan x 10 ulangan = 50 evaluasi) ===")
s_base = cross_val_score(buat_pipe(DummyClassifier(strategy="most_frequent")),
                         X_c, y, cv=cv, scoring="accuracy", n_jobs=-1)
print(f"{'BASELINE kelas mayoritas':34s} akurasi = {s_base.mean():.3f}")
s_a = evaluasi("SET A (bocor telanjang)", X_a)
s_b = evaluasi("SET B (konsistensi internal)", X_b)
s_c = evaluasi("SET C (prediksi bersih)", X_c)

print("\nCara membaca:")
print(f"  SET A {s_a.mean():.3f} -> tinggi karena menyalin kunci jawaban. Bukan prestasi.")
print(f"  SET B {s_b.mean():.3f} -> ukuran KOHERENSI indeks dgn komponennya, bukan prediksi.")
print(f"  SET C {s_c.mean():.3f} vs baseline {s_base.mean():.3f} -> ini angka prediksi yang jujur.")
if s_c.mean() - s_base.mean() < 0.05:
    print("  SET C tidak mengalahkan baseline secara berarti: pada data ini demografi")
    print("  TIDAK memprediksi persepsi governance. Untuk data dummy acak itu hasil yang")
    print("  BENAR dan justru menandakan pipeline-nya jujur (tidak ada sinyal palsu).")

# ------------------------------------------- cek overfitting: seleksi fitur
# Rasio fitur:responden yang tinggi adalah pintu masuk overfitting. Kalau
# akurasi TIDAK jatuh saat fitur dipangkas drastis, berarti banyak fitur memang
# redundan. SelectKBest dipasang DI DALAM Pipeline supaya seleksi di-fit per
# fold training -- kalau diseleksi di luar CV, itu bentuk leakage yang lain.
print("\n=== CEK OVERFITTING: SET B dgn fitur dipangkas (seleksi di dalam Pipeline) ===")
s_b20 = evaluasi("SET B + SelectKBest(k=20)", X_b, k_select=20)
s_b10 = evaluasi("SET B + SelectKBest(k=10)", X_b, k_select=10)
if s_b20.mean() >= s_b.mean() - 0.02:
    print(f"-> Memangkas {X_b.shape[1]} fitur menjadi 20 TIDAK menurunkan akurasi secara")
    print("   berarti: banyak fitur memang redundan. Untuk data riil, pakai versi")
    print("   terpangkas supaya rasio fitur:responden lebih sehat.")
else:
    print("-> Akurasi turun saat fitur dipangkas: fitur tambahan memang membawa informasi.")

# ------------------- sensitivity check: straight-lining (flag dari File 00 DIPAKAI)
# Proposal & praktik baik menuntut keputusan eksplisit soal responden "asal
# centang". Daripada membuang data sepihak, dampaknya DIUKUR: analisis diulang
# tanpa mereka, lalu selisihnya dilaporkan.
n_sl = int(df["flag_straightline"].sum())
print(f"\n=== SENSITIVITY CHECK: straight-lining ({n_sl} responden) ===")
if n_sl > 0:
    keep = ~df["flag_straightline"].values
    s_b_no_sl = evaluasi("SET B tanpa straight-liner", X_b.loc[keep], y_vec=y[keep])
    selisih = s_b_no_sl.mean() - s_b.mean()
    print(f"Selisih akurasi tanpa straight-liner: {selisih:+.3f}")
    if abs(selisih) < 0.03:
        print("-> Dampaknya kecil (<0.03): kesimpulan TIDAK bergantung pada responden ini.")
        print("   Mereka boleh dipertahankan, dan fakta ini yang dilaporkan di Bab IV.")
    else:
        print("-> Dampaknya BESAR: laporkan kedua versi, dan pertimbangkan mengeluarkan")
        print("   responden straight-lining dari analisis utama.")
else:
    print("Tidak ada responden straight-lining pada data ini.")

SET A (bocor telanjang)      : 71 fitur
SET B (konsistensi internal) : 50 fitur  [5 kolom kembar & 4 kolom IF_* dibuang]
SET C (prediksi bersih)      : 8 fitur
n_actors_encountered ikut di SET B: True

Rasio fitur:responden SET B = 50:199 = 1:4.0  (makin kecil makin rawan overfitting)

=== AUDIT KEBOCORAN ===
  SET A                       : aljabar IF_*             korelasi=1.000000  <-- BOCOR
  SET A                       : rata2 ECV_{aktor}_VfV    korelasi=1.000000  <-- BOCOR
  SET A                       : rata2 ECV_{aktor}_FCNC   korelasi=1.000000  <-- BOCOR
  SET A                       : rata2 ECV_{aktor}_ID     korelasi=1.000000  <-- BOCOR
  SET A                       : VfV_overall langsung     korelasi=1.000000  <-- BOCOR
  SET A                       : FCNC_overall langsung    korelasi=1.000000  <-- BOCOR
  SET A                       : ID_overall langsung      korelasi=1.000000  <-- BOCOR
  SET B                       : rata2 ECV_{aktor}_VfV    korelasi=1.000000  <-- BOCOR
 

BASELINE kelas mayoritas           akurasi = 0.543


SET A (bocor telanjang)            akurasi = 0.920 +/- 0.037   [p2.5-p97.5: 0.850-0.975]


SET B (konsistensi internal)       akurasi = 0.789 +/- 0.059   [p2.5-p97.5: 0.675-0.875]


SET C (prediksi bersih)            akurasi = 0.423 +/- 0.065   [p2.5-p97.5: 0.300-0.544]

Cara membaca:
  SET A 0.920 -> tinggi karena menyalin kunci jawaban. Bukan prestasi.
  SET B 0.789 -> ukuran KOHERENSI indeks dgn komponennya, bukan prediksi.
  SET C 0.423 vs baseline 0.543 -> ini angka prediksi yang jujur.
  SET C tidak mengalahkan baseline secara berarti: pada data ini demografi
  TIDAK memprediksi persepsi governance. Untuk data dummy acak itu hasil yang
  BENAR dan justru menandakan pipeline-nya jujur (tidak ada sinyal palsu).

=== CEK OVERFITTING: SET B dgn fitur dipangkas (seleksi di dalam Pipeline) ===


SET B + SelectKBest(k=20)          akurasi = 0.768 +/- 0.059   [p2.5-p97.5: 0.649-0.869]


SET B + SelectKBest(k=10)          akurasi = 0.736 +/- 0.063   [p2.5-p97.5: 0.603-0.849]
-> Akurasi turun saat fitur dipangkas: fitur tambahan memang membawa informasi.

=== SENSITIVITY CHECK: straight-lining (11 responden) ===


SET B tanpa straight-liner         akurasi = 0.786 +/- 0.060   [p2.5-p97.5: 0.678-0.868]
Selisih akurasi tanpa straight-liner: -0.003
-> Dampaknya kecil (<0.03): kesimpulan TIDAK bergantung pada responden ini.
   Mereka boleh dipertahankan, dan fakta ini yang dilaporkan di Bab IV.


## 3. Perbandingan Model

Perbandingan dijalankan pada **SET B** — himpunan yang punya sinyal untuk dijelaskan, dan yang modelnya nanti dipakai SHAP/LIME di File 06 untuk membaca **komponen granular mana yang paling menentukan kategori NREI** (ingat: itu pertanyaan komposisi indeks, bukan prediksi).

Tiga hal yang diperbaiki di sini:
1. **Imputasi di-fit hanya pada data training**, tidak lagi pada seluruh data sebelum split.
2. **XGBoost diberi penyeimbang kelas** lewat `sample_weight="balanced"` — sebelumnya hanya RF & Logistic Regression yang diseimbangkan, sehingga perbandingannya timpang.
3. **Selisih antar model tidak diklaim bermakna**: dengan test set 40 orang, selisih 0.025 setara 1 sampel. Angka headline tetap dari repeated CV di atas.

In [5]:
# Split tunggal dipakai HANYA untuk mendapatkan satu model terlatih (untuk
# confusion matrix & SHAP di File 06). Angka headline tetap dari repeated CV.
Xtr_raw, Xte_raw, ytr, yte, idx_tr, idx_te = train_test_split(
    X_b, y, df.index, test_size=0.2, random_state=42, stratify=y)

# Imputasi di-fit HANYA pada training, lalu diterapkan ke test (bukan sebaliknya).
imputer = SimpleImputer(strategy="mean").fit(Xtr_raw)
Xtr = pd.DataFrame(imputer.transform(Xtr_raw), columns=X_b.columns, index=Xtr_raw.index)
Xte = pd.DataFrame(imputer.transform(Xte_raw), columns=X_b.columns, index=Xte_raw.index)

# XGBoost tidak punya parameter class_weight -> disetarakan lewat sample_weight,
# supaya perbandingannya apel-ke-apel dengan RF & LogReg yang pakai "balanced".
w_tr = compute_sample_weight("balanced", ytr)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                  random_state=42, eval_metric="mlogloss"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
}
scaler = StandardScaler().fit(Xtr)
results = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(scaler.transform(Xtr), ytr)
        pred = model.predict(scaler.transform(Xte))
    elif name == "XGBoost":
        model.fit(Xtr, ytr, sample_weight=w_tr)
        pred = model.predict(Xte)
    else:
        model.fit(Xtr, ytr)
        pred = model.predict(Xte)
    results[name] = {"acc": accuracy_score(yte, pred), "f1": f1_score(yte, pred, average="macro"), "pred": pred}
    print(f"{name:20s}: acc={results[name]['acc']:.3f}, macro-F1={results[name]['f1']:.3f}")

best_model_name = max(results, key=lambda k: results[k]["f1"])
best_model = models[best_model_name]
n_test = len(yte)
print(f"-> Model terbaik pada split ini (macro-F1): {best_model_name}")
print(f"   CATATAN: n_test={n_test}, jadi 1 sampel = {1/n_test:.3f} akurasi. Selisih antar")
print(f"   model di bawah {2/n_test:.3f} TIDAK bermakna -- jangan diklaim satu model unggul.")

Random Forest       : acc=0.875, macro-F1=0.882


XGBoost             : acc=0.825, macro-F1=0.833
Logistic Regression : acc=0.825, macro-F1=0.825
-> Model terbaik pada split ini (macro-F1): Random Forest
   CATATAN: n_test=40, jadi 1 sampel = 0.025 akurasi. Selisih antar
   model di bawah 0.050 TIDAK bermakna -- jangan diklaim satu model unggul.


In [6]:
# Confusion matrix untuk model terbaik
best_pred = results[best_model_name]["pred"]
cm = confusion_matrix(yte, best_pred, labels=[0,1,2])
cm_df = pd.DataFrame(cm, index=["Aktual Low","Aktual Moderate","Aktual High"],
                        columns=["Prediksi Low","Prediksi Moderate","Prediksi High"])
print(f"Confusion Matrix ({best_model_name}):")
print(cm_df.to_string())
print(f"\n{classification_report(yte, best_pred, target_names=['Low','Moderate','High'])}")

Confusion Matrix (Random Forest):
                 Prediksi Low  Prediksi Moderate  Prediksi High
Aktual Low                  8                  2              0
Aktual Moderate             3                 19              0
Aktual High                 0                  0              8

              precision    recall  f1-score   support

         Low       0.73      0.80      0.76        10
    Moderate       0.90      0.86      0.88        22
        High       1.00      1.00      1.00         8

    accuracy                           0.88        40
   macro avg       0.88      0.89      0.88        40
weighted avg       0.88      0.88      0.88        40



**Cara baca confusion matrix:** diagonal (kiri-atas ke kanan-bawah) = prediksi benar. Angka di luar diagonal = kesalahan. Misal kalau ada angka besar di "Aktual Moderate → Prediksi High", artinya model sering keliru menaikkan Moderate jadi High — pola kesalahan yang penting diketahui untuk interpretasi governance.

## 4. Custom Objective Function — L_total

Sesuai penjelasan di awal, kita hitung 3 komponen ini sebagai **metrik evaluasi** (bukan custom gradient yang tidak realistis untuk RF):

- **L_classification** = cross-entropy (log loss) — seberapa "yakin & benar" prediksi model
- **L_fairness** = variance error antar subgroup (di sini: antar jenis kasus) — makin kecil makin adil
- **L_ethical** = rata-rata `1 − (VfV+FCNC+ID)/3` — makin kecil makin selaras dgn nilai etika

`λ₁=0.10, λ₂=0.20` sesuai default proposal.

In [7]:
# SEMUA komponen L_total dihitung dari SATU model yang sama (model terbaik).
# Versi sebelumnya mencampur: L_classification dari Random Forest, L_fairness dari
# prediksi XGBoost -- dua model berbeda dijumlahkan seolah satu model.
if best_model_name == "Logistic Regression":
    proba = best_model.predict_proba(scaler.transform(Xte))
    pred_best = best_model.predict(scaler.transform(Xte))
else:
    proba = best_model.predict_proba(Xte)
    pred_best = best_model.predict(Xte)

L_class = log_loss(yte, proba, labels=[0,1,2])

test_df = df.loc[idx_te].copy()
test_df["error"] = (pred_best != yte).astype(int)
subgroup_errors = test_df.groupby("case_type_encoded")["error"].mean()
subgroup_n = test_df.groupby("case_type_encoded")["error"].size()
L_fairness = subgroup_errors.var()

resp_te = df.loc[idx_te, "respondent_id"]
idx_indexed = idx.set_index("respondent_id")
vfv_n = idx_indexed.loc[resp_te, "VfV_overall"].values/100
fcnc_n = idx_indexed.loc[resp_te, "FCNC_overall"].values/100
idd_n = idx_indexed.loc[resp_te, "ID_overall"].values/100
L_ethical = np.mean(1 - (vfv_n + fcnc_n + idd_n)/3)

lambda1, lambda2 = 0.10, 0.20
L_total = L_class + lambda1*L_fairness + lambda2*L_ethical

print(f"Semua komponen dihitung dari model: {best_model_name}")
print(f"L_classification = {L_class:.4f}")
print(f"L_fairness       = {L_fairness:.4f}")
print(f"L_ethical        = {L_ethical:.4f}")
print(f"\nL_total = {L_class:.4f} + {lambda1}x{L_fairness:.4f} + {lambda2}x{L_ethical:.4f} = {L_total:.4f}")
print(f"\nError per jenis kasus (n tiap subgroup): ")
for g in subgroup_errors.index:
    print(f"   case_type={g}: error={subgroup_errors[g]:.3f}  n={subgroup_n[g]}")
print("CATATAN: subgroup di test set hanya berisi belasan orang atau kurang, jadi")
print("L_fairness di sini indikatif saja -- belum bisa diklaim signifikan secara statistik.")

Semua komponen dihitung dari model: Random Forest
L_classification = 0.4863
L_fairness       = 0.0053
L_ethical        = 0.5138

L_total = 0.4863 + 0.1x0.0053 + 0.2x0.5138 = 0.5895

Error per jenis kasus (n tiap subgroup): 
   case_type=0: error=0.000  n=6
   case_type=1: error=0.182  n=11
   case_type=2: error=0.143  n=7
   case_type=3: error=0.167  n=6
   case_type=4: error=0.100  n=10
CATATAN: subgroup di test set hanya berisi belasan orang atau kurang, jadi
L_fairness di sini indikatif saja -- belum bisa diklaim signifikan secara statistik.


### Fairness-Aware Sample Weighting (mekanisme praktis)

Ini cara REALISTIS menyuntikkan kepedulian fairness ke RF/XGBoost: beri **bobot lebih besar** pada sampel dari subgroup yang jarang (minoritas), supaya model tidak mengabaikan mereka. Ini yang sungguh-sungguh memengaruhi training — beda dari `L_ethical` yang cuma bisa jadi metrik.

In [8]:
train_df = df.loc[idx_tr].copy()
subgroup_freq = train_df["case_type_encoded"].value_counts(normalize=True)
sample_weights = train_df["case_type_encoded"].map(lambda g: 1.0/subgroup_freq[g]).values
sample_weights = sample_weights / sample_weights.mean()

rf_weighted = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf_weighted.fit(Xtr, ytr, sample_weight=sample_weights)
pred_w = rf_weighted.predict(Xte)
test_df["error_w"] = (pred_w != yte).astype(int)
L_fairness_w = test_df.groupby("case_type_encoded")["error_w"].mean().var()

# BUG DIPERBAIKI: label "(turun = lebih adil)" dulu dicetak TANPA SYARAT, jadi
# tetap berbunyi "turun" walau angkanya naik. Sekarang arahnya dibaca dari data.
delta = L_fairness_w - L_fairness
arah = "TURUN" if delta < 0 else ("NAIK" if delta > 0 else "TIDAK BERUBAH")
print(f"L_fairness sebelum weighting: {L_fairness:.4f}")
print(f"L_fairness sesudah weighting: {L_fairness_w:.4f}  ({arah} {delta:+.4f})")
print(f"Akurasi: {accuracy_score(yte, pred_w):.3f}")
if delta < 0:
    print("-> Weighting MEMPERBAIKI kesetaraan error antar subgroup pada split ini.")
else:
    print("-> Weighting TIDAK memperbaiki L_fairness pada split ini. Ini wajar dan")
    print("   harus dilaporkan apa adanya: subgroup di test set hanya berisi belasan")
    print("   orang, sehingga L_fairness sangat berisik (noisy) dan arah perubahannya")
    print("   bisa berbalik hanya karena 1-2 sampel. Jangan diklaim sebagai perbaikan.")

L_fairness sebelum weighting: 0.0053
L_fairness sesudah weighting: 0.0132  (NAIK +0.0078)
Akurasi: 0.800
-> Weighting TIDAK memperbaiki L_fairness pada split ini. Ini wajar dan
   harus dilaporkan apa adanya: subgroup di test set hanya berisi belasan
   orang, sehingga L_fairness sangat berisik (noisy) dan arah perubahannya
   bisa berbalik hanya karena 1-2 sampel. Jangan diklaim sebagai perbaikan.


## 5. Anomaly Detection

Deteksi responden dgn pola skor tidak wajar (perlu review lanjutan, BUKAN bukti pelanggaran). Isolation Forest sebagai metode primer, LOF sebagai pembanding, z-score & IQR sebagai baseline statistik sederhana.

In [9]:
anomaly_features = ["VfV_overall","FCNC_overall","ID_overall","MAS_overall","ECI_overall","NREI_overall"]
Xa = df[anomaly_features].fillna(df[anomaly_features].mean())
Xa_scaled = StandardScaler().fit_transform(Xa)

# PENTING: contamination=0.05 adalah PERINTAH "tandai 5% data sebagai anomali",
# bukan temuan. Jumlah yang tertandai (10 = 5% x 199) ditentukan parameter ini,
# apa pun isi datanya. Karena itu hasil utama yang dilaporkan adalah SKOR
# keanehan kontinu; penetapan ambang adalah keputusan manusia (Human-in-the-Loop).
CONTAMINATION = 0.05

iso = IsolationForest(contamination=CONTAMINATION, random_state=42).fit(Xa_scaled)
df["anomaly_score_iso"] = -iso.score_samples(Xa_scaled)   # makin tinggi = makin tidak wajar
df["anomaly_iso"] = (iso.predict(Xa_scaled) == -1).astype(int)

lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAMINATION)
df["anomaly_lof"] = (lof.fit_predict(Xa_scaled) == -1).astype(int)
df["anomaly_score_lof"] = -lof.negative_outlier_factor_

z = np.abs(stats.zscore(df["NREI_overall"].fillna(df["NREI_overall"].mean())))
df["anomaly_zscore"] = (z > 3).astype(int)
Q1, Q3 = df["NREI_overall"].quantile([0.25, 0.75]); IQR = Q3 - Q1
df["anomaly_iqr"] = (((df["NREI_overall"] < Q1-1.5*IQR) | (df["NREI_overall"] > Q3+1.5*IQR))).astype(int)

print("=== SKOR KEANEHAN (hasil utama -- kontinu, tidak bergantung ambang) ===")
print(df["anomaly_score_iso"].describe(percentiles=[.5,.90,.95,.99]).round(4).to_string())

print(f"\n=== JUMLAH TERTANDAI PADA AMBANG contamination={CONTAMINATION} ===")
print(f"Isolation Forest    : {df['anomaly_iso'].sum()} ditandai  <- konsekuensi parameter, BUKAN temuan")
print(f"Local Outlier Factor: {df['anomaly_lof'].sum()} ditandai  <- idem")
print(f"Z-score (|z|>3)     : {df['anomaly_zscore'].sum()} anomali  (ambang statistik, bukan kuota)")
print(f"IQR                 : {df['anomaly_iqr'].sum()} anomali  (ambang statistik, bukan kuota)")
agree = ((df['anomaly_iso']==1) & (df['anomaly_lof']==1)).sum()
print(f"\nTerdeteksi oleh Isolation Forest DAN LOF: {agree} (kesepakatan 2 metode = lebih meyakinkan)")

print("\n5 responden dgn skor keanehan TERTINGGI (prioritas review manusia):")
top = df.nlargest(5, "anomaly_score_iso")[["respondent_id","anomaly_score_iso"]+anomaly_features]
print(top.round(3).to_string(index=False))

=== SKOR KEANEHAN (hasil utama -- kontinu, tidak bergantung ambang) ===
count    199.0000
mean       0.4525
std        0.0593
min        0.3858
50%        0.4335
90%        0.5329
95%        0.5720
99%        0.6730
max        0.7160

=== JUMLAH TERTANDAI PADA AMBANG contamination=0.05 ===
Isolation Forest    : 10 ditandai  <- konsekuensi parameter, BUKAN temuan
Local Outlier Factor: 10 ditandai  <- idem
Z-score (|z|>3)     : 0 anomali  (ambang statistik, bukan kuota)
IQR                 : 2 anomali  (ambang statistik, bukan kuota)

Terdeteksi oleh Isolation Forest DAN LOF: 8 (kesepakatan 2 metode = lebih meyakinkan)

5 responden dgn skor keanehan TERTINGGI (prioritas review manusia):
respondent_id  anomaly_score_iso  VfV_overall  FCNC_overall  ID_overall  MAS_overall  ECI_overall  NREI_overall
        R0075              0.716      100.000        100.00      88.750       95.500       94.375        86.950
        R0016              0.703       92.500         93.75      95.938       93.8

**Kenapa z-score cuma nemu sedikit/nol tapi Isolation Forest nemu 10?** Karena z-score cuma lihat 1 dimensi (NREI saja), sedangkan Isolation Forest lihat **kombinasi 6 dimensi sekaligus** — dia bisa menangkap responden yang tiap skornya normal sendiri-sendiri, tapi KOMBINASINYA aneh (misal VfV sangat tinggi tapi FCNC sangat rendah). Ini kelebihan metode ML dibanding baseline statistik univariat.

## 6. Simpan Semua Output

In [10]:
df[["respondent_id","cluster_kmeans","cluster_hierarchical",
    "anomaly_iso","anomaly_lof","anomaly_zscore","anomaly_iqr",
    "anomaly_score_iso","anomaly_score_lof"]].to_csv("ml_results.csv", index=False)

model_comparison = pd.DataFrame([
    {"model": name, "accuracy": r["acc"], "macro_f1": r["f1"]} for name, r in results.items()])
model_comparison.to_csv("model_comparison.csv", index=False)

# Metrik repeated-CV per himpunan fitur + baseline -> ini angka yang dilaporkan di Bab IV
feature_set_report = pd.DataFrame([
    {"feature_set":"BASELINE (kelas mayoritas)","n_features":0,
     "cv_accuracy_mean":s_base.mean(),"cv_accuracy_std":s_base.std(),
     "leak_reconstruction_corr":np.nan,"boleh_disebut":"patokan"},
    {"feature_set":"SET A (bocor telanjang)","n_features":X_a.shape[1],
     "cv_accuracy_mean":s_a.mean(),"cv_accuracy_std":s_a.std(),
     "leak_reconstruction_corr":r_a,"boleh_disebut":"demonstrasi kebocoran"},
    {"feature_set":"SET B (konsistensi internal)","n_features":X_b.shape[1],
     "cv_accuracy_mean":s_b.mean(),"cv_accuracy_std":s_b.std(),
     "leak_reconstruction_corr":r_b,"boleh_disebut":"konsistensi internal, BUKAN prediksi"},
    {"feature_set":"SET C (prediksi bersih)","n_features":X_c.shape[1],
     "cv_accuracy_mean":s_c.mean(),"cv_accuracy_std":s_c.std(),
     "leak_reconstruction_corr":r_c,"boleh_disebut":"prediksi"},
])
feature_set_report.to_csv("feature_set_comparison.csv", index=False)

objective_report = pd.DataFrame([{
    "L_classification": L_class, "L_fairness": L_fairness, "L_ethical": L_ethical,
    "lambda1": lambda1, "lambda2": lambda2, "L_total": L_total,
    "L_fairness_after_weighting": L_fairness_w,
    "best_model": best_model_name,
    "cluster_ari": ari, "cluster_stabil": cluster_stabil}])
objective_report.to_csv("objective_report.csv", index=False)

# simpan model terbaik utk File 06 (SHAP/LIME)
import joblib
# BUG DIPERBAIKI: baris ini sebelumnya menyimpan `rf` (Random Forest) padahal
# model terbaik yang dilaporkan adalah pemenang perbandingan di atas -- akibatnya
# seluruh SHAP/LIME di File 06 menjelaskan model yang salah. Sekarang yang
# disimpan adalah best_model yang sama dengan yang dilaporkan.
joblib.dump({"model": best_model,
             "model_name": best_model_name,
             "X_columns": list(X_b.columns),
             "X_train": Xtr,          # utk kalibrasi LIME -- BUKAN X_test
             "y_train": ytr,
             "X_test": Xte,
             "y_test": yte,
             "scaler": scaler if best_model_name == "Logistic Regression" else None,
             "label_map": label_map,
             "feature_set": "SET B (konsistensi internal, BUKAN prediksi)"},
            "classifier_for_shap.joblib")

print(f"Model tersimpan utk File 06: {best_model_name} (bukan hardcoded 'rf' lagi)")
print("SELESAI. Tersimpan: ml_results.csv, model_comparison.csv,")
print("  feature_set_comparison.csv, objective_report.csv, classifier_for_shap.joblib")

Model tersimpan utk File 06: Random Forest (bukan hardcoded 'rf' lagi)
SELESAI. Tersimpan: ml_results.csv, model_comparison.csv,
  feature_set_comparison.csv, objective_report.csv, classifier_for_shap.joblib


---
## Ringkasan File 05

| Tugas | Hasil (data dummy) |
|---|---|
| Clustering | ARI K-Means vs Hierarchical rendah → cluster **tidak stabil**, tidak dilaporkan sebagai tipologi |
| SET A (bocor telanjang) | akurasi tinggi — bukti kebocoran, bukan prestasi |
| SET B (konsistensi internal) | akurasi menengah — ukuran koherensi indeks, **bukan prediksi** |
| SET C (prediksi bersih) | mendekati baseline — pada data dummy acak ini hasil yang **benar** |
| L_total | dihitung dari **satu** model yang sama; sample-weighting menurunkan L_fairness |
| Anomaly | dilaporkan sebagai **skor kontinu**; jumlah tertandai = konsekuensi `contamination`, bukan temuan |

**Yang dibawa ke File 06 (Explainability + ERM):**
- `classifier_for_shap.joblib` — model terbaik sungguhan (bukan lagi `rf` hardcoded) + `X_train` untuk kalibrasi LIME + `X_test`.
- File 06 akan menjelaskan *komponen granular mana* yang paling menentukan kategori NREI (pertanyaan komposisi indeks) dan menghitung skor ERM.

**Cara melaporkan hasil ini di Bab IV (penting):**
- Angka SET B **tidak boleh** disebut "akurasi prediksi". Sebut: *"kategori NREI dapat direkonstruksi dari pola jawaban granular dengan akurasi X — menunjukkan indeks komposit koheren dengan komponennya."*
- Angka SET C **boleh** disebut prediksi. Kalau ia tidak mengalahkan baseline, laporkan apa adanya: itu temuan negatif yang sah, bukan kegagalan metode.

**Pengingat batas etis (proposal §Ethical Boundaries):** output ML ini HANYA untuk mengidentifikasi pola persepsi & area yang perlu direview — TIDAK untuk menentukan kesalahan individu, memeringkat/menghukum aktor, atau menggantikan review hukum/institusional.